In [0]:
# DAY2
#PySpark code to find duplicate records with dummy data

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count

spark = SparkSession.builder.appName("FindDuplicates").getOrCreate()

data = [
    (1, "Ravi", "IT", 50000),
    (2, "Anu", "HR", 60000),
    (3, "Ravi", "IT", 50000),
    (4, "Meera", "Finance", 70000),
    (5, "Anu", "HR", 60000)
]

columns = ["emp_id", "emp_name", "department", "salary"]
df = spark.createDataFrame(data, columns)
df.show()
df_duplicate = df.groupBy("emp_name", "department", "salary") \
    .count() \
    .filter(col("count") > 1)
df_duplicate.show()

#To display full duplicate rows
df_duplicate_rows =( 
    df.join(
        df_duplicate.select("emp_name", "department", "salary"), 
        on=["emp_name", "department", "salary"],
        how="inner"
    )
)
df_duplicate_rows.show()

# DAY3
# Remove the all the duplicate rows
df_no_duplicates = df.dropDuplicates(["emp_name", "department", "salary"])
df_no_duplicates.show()

# Another method
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number
window_spec =(
     Window.partitionBy("emp_name", "department", "salary").orderBy(col("emp_id"))
)
df_no_duplicates = (
     df.withColumn("row_number", row_number().over(window_spec))
     .filter(col("row_number") == 1)
     .drop(col("row_number"))
)
df_no_duplicates.show()

In [0]:
#DAY4 Top 3 salaries for Each department
from pyspark.sql.window import Window
from pyspark.sql.functions import col, row_number
data = [
    ("HR", "A", 5000),
    ("HR", "B", 6000),
    ("HR", "C", 7000),
    ("HR", "D", 8000),
    ("IT", "E", 9000),
    ("IT", "F", 10000),
    ("IT", "G", 11000),
    ("IT", "H", 12000),
    ("Finance", "I", 4000),
    ("Finance", "J", 4500),
    ("Finance", "K", 4800),
    ("Finance", "L", 3000)
]

columns = ["department", "employee", "salary"]

df = spark.createDataFrame(data, columns)
df.show()
window_spec = Window.partitionBy("department").orderBy(col("salary").desc())
df_ranked = df.withColumn("rank", row_number().over(window_spec))
top_3 = df_ranked.filter(col("rank") <=3)
top_3.show()

In [0]:

#DAY5 JOINS
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

# Initialize Spark
spark = SparkSession.builder.appName("All_Join_Conditions").getOrCreate()

# -------------------------
# Create Table A & Table B
# -------------------------
data_a = [(1,), (1,), (1,), (0,), (3,)]
data_b = [(1,), (0,), (2,)]

df_a = spark.createDataFrame(data_a, ["value"])
df_b = spark.createDataFrame(data_b, ["value"])

a = df_a.alias("a")
b = df_b.alias("b")

print("===== TABLE A =====")
df_a.show()

print("===== TABLE B =====")
df_b.show()

# -------------------------
# 1. INNER JOIN
# -------------------------
print("===== INNER JOIN =====")
inner_df = a.join(b, col("a.value") == col("b.value"), "inner") \
    .select(col("a.value").alias("a_value"), col("b.value").alias("b_value"))
inner_df.show()
print("Count =", inner_df.count())

# -------------------------
# 2. LEFT JOIN
# -------------------------
print("===== LEFT JOIN =====")
left_df = a.join(b, col("a.value") == col("b.value"), "left") \
    .select(col("a.value").alias("a_value"), col("b.value").alias("b_value"))
left_df.show()
print("Count =", left_df.count())

# -------------------------
# 3. RIGHT JOIN
# -------------------------
print("===== RIGHT JOIN =====")
right_df = a.join(b, col("a.value") == col("b.value"), "right") \
    .select(col("a.value").alias("a_value"), col("b.value").alias("b_value"))
right_df.show()
print("Count =", right_df.count())

# -------------------------
# 4. FULL OUTER JOIN
# -------------------------
print("===== FULL OUTER JOIN =====")
full_df = a.join(b, col("a.value") == col("b.value"), "outer") \
    .select(col("a.value").alias("a_value"), col("b.value").alias("b_value"))
full_df.show()
print("Count =", full_df.count())

# -------------------------
# 5. CROSS JOIN
# -------------------------
print("===== CROSS JOIN =====")
cross_df = a.crossJoin(b) \
    .select(col("a.value").alias("a_value"), col("b.value").alias("b_value"))
cross_df.show()
print("Count =", cross_df.count())

# -------------------------
# 6. LEFT ANTI JOIN
# -------------------------
print("===== LEFT ANTI JOIN =====")
left_anti_df = a.join(b, col("a.value") == col("b.value"), "left_anti") \
    .select(col("a.value").alias("a_value"))
left_anti_df.show()
print("Count =", left_anti_df.count())

# -------------------------
# 7. RIGHT ANTI JOIN
# -------------------------
print("===== RIGHT ANTI JOIN =====")
right_anti_df = b.join(a, col("b.value") == col("a.value"), "left_anti") \
    .select(col("b.value").alias("b_value"))
right_anti_df.show()
print("Count =", right_anti_df.count())

# -------------------------
# 8. LEFT SEMI JOIN
# -------------------------
print("===== LEFT SEMI JOIN =====")
left_semi_df = a.join(b, col("a.value") == col("b.value"), "left_semi") \
    .select(col("a.value").alias("a_value"))
left_semi_df.show()
print("Count =", left_semi_df.count())

# -------------------------
# 9. RIGHT SEMI JOIN
# -------------------------
print("===== RIGHT SEMI JOIN =====")
right_semi_df = b.join(a, col("b.value") == col("a.value"), "left_semi") \
    .select(col("b.value").alias("b_value"))
right_semi_df.show()
print("Count =", right_semi_df.count())

# Stop Spark
spark.stop()

In [0]:
#DAY6 Case Statement Example  if the salary is greater than 10000 , mention high --50000 to 100000 Medium
from pyspark.sql import SparkSession
from pyspark.sql.functions import when, col

# Initialize Spark
spark = SparkSession.builder.appName("Case_Statement").getOrCreate()

# Sample Data (dummy employees table)
data = [
    ("Ravi", 120000),
    ("Anu", 75000),
    ("Kiran", 40000),
    ("Meera", 100000),
    ("John", 50000)
]

df = spark.createDataFrame(data, ["emp_name", "salary"])

# CASE WHEN logic in PySpark
result_df = df.withColumn(
    "category",
    when(col("salary") > 100000, "High")
    .when((col("salary") >= 50000) & (col("salary") <= 100000), "Medium")
    .otherwise("Low")
)

# Show result
result_df.show()

In [0]:
#DAY7 Running total
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.functions import sum as spark_sum, col

spark = SparkSession.builder.appName("Running_Total_Department_Wise").getOrCreate()

# Sample Employee Data
data = [
    (104, "Priya", 1, 70000),
    (105, "Kiran", 1, 50000),
    (101, "Amit", 2, 90000),
    (102, "Neha", 2, 80000),
    (103, "Ravi", 2, 75000),
    (106, "Arjun", 3, 60000),
    (107, "Meena", 3, 55000),
    (108, "Suresh", 4, 65000),
    (109, "Divya", 4, 48000),
    (110, "Rahul", 4, 47000)
]

columns = ["emp_id", "emp_name", "dept_id", "salary"]

employees = spark.createDataFrame(data, columns)
window_spec = Window.partitionBy("dept_id").orderBy(col("emp_id"))
running_df_total = employees.withColumn("running_total", spark_sum("salary").over(window_spec))
running_df_total.show()

In [0]:
#DAY8
from pyspark.sql import SparkSession
from pyspark.sql.functions import sum as spark_sum, row_number, col, to_date
from pyspark.sql.window import Window

# Create Spark Session
spark = SparkSession.builder.appName("Orders_Aggregations").getOrCreate()

# Dummy Data
data = [
    (1001, 1, "2024-01-05", 250.00),
    (1002, 2, "2024-01-06", 450.00),
    (1003, 1, "2024-01-10", 300.00),
    (1004, 3, "2024-01-12", 150.00),
    (1005, 2, "2024-01-15", 700.00),
    (1006, 4, "2024-01-18", 200.00),
    (1007, 1, "2024-01-20", 500.00),
    (1008, 3, "2024-01-25", 350.00),
    (1009, 5, "2024-02-01", 600.00),
    (1010, 2, "2024-02-05", 800.00),
    (1011, 4, "2024-02-08", 120.00),
    (1012, 1, "2024-02-10", 900.00),
    (1013, 3, "2024-02-12", 400.00),
    (1014, 5, "2024-02-15", 1000.00),
    (1015, 2, "2024-02-18", 650.00),
    (1016, 6, "2024-02-20", 300.00),
    (1017, 4, "2024-02-22", 220.00),
    (1018, 1, "2024-02-25", 750.00),
    (1019, 3, "2024-02-27", 500.00),
    (1020, 5, "2024-03-01", 850.00)
]

columns = ["order_id", "customer_id", "order_date", "amount"]

orders = spark.createDataFrame(data, columns)

# Convert order_date string to date type
orders = orders.withColumn("order_date", to_date(col("order_date")))

orders.show()

df_total_amt = orders.groupBy("customer_id").agg(spark_sum("amount").alias("total_amount")).orderBy("customer_id")
df_total_amt.show()
#Running total
window_spec = Window.partitionBy("customer_id").orderBy("order_date")
running_df_total = orders.withColumn("running_total", spark_sum("amount").over(window_spec))
running_df_total.show()

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

# Create Spark Session
spark = SparkSession.builder.appName("Employee_Manager_Salary").getOrCreate()

# Dummy Data
data = [
    (1, "Ravi", 90000, None),      # Manager
    (2, "Anu", 95000, 1),          # Earns more than Ravi
    (3, "Kiran", 70000, 1),
    (4, "Meera", 120000, None),    # Manager
    (5, "John", 110000, 4),
    (6, "Priya", 130000, 4),       # Earns more than Meera
    (7, "Arjun", 60000, 3)
]

columns = ["emp_id", "emp_name", "salary", "manager_id"]

# Create DataFrame
employees = spark.createDataFrame(data, columns)

print("===== Employee Data =====")
employees.show()
#self join
e = employees.alias("e")
m = employees.alias("m")

result_df = e.join(m, col("e.manager_id") == col("m.emp_id"), "inner").filter(col("e.salary") > col("m.salary")) \
    .select( col("e.emp_id").alias("emp_id"),
             col("e.emp_name").alias("emp_name"), 
             col("e.salary") .alias("emp_salary"),
             col("m.emp_name").alias("manager_name"), 
             col("m.salary").alias("manager_salary"))
result_df.show()